In [1]:
import pandas as pd 
import matplotlib.pyplot as plt
import matplotlib.dates as mdates#Date Parser

import seaborn as sns
sns.set_style('white')
sns.set(rc={'figure.figsize':(11, 4)})

In [2]:
import os
from pathlib import Path

In [16]:
DOWNLOAD = False

урок вдохновлён данным репозиторием https://github.com/jenfly/opsd и проектом https://www.machinelearningplus.com/time-series/time-series-analysis-python/

В качестве  набор данных для практики рассмотрим часть набор данных [открытые данные энергетических систем](https://open-power-system-data.org/). Мы будем работать с данным относящимися ко временным рядам (https://data.open-power-system-data.org/time_series/). По приведенной ссылке можно найти описание набора данных. Также набор можно найти на странице официального репозитория: https://github.com/Open-Power-System-Data/time_series.

Для начала давайте попробуем загрузить последнюю версию набора данных.

In [17]:
if DOWNLOAD:
    # Download hourly data from OPSD website
    url = 'https://data.open-power-system-data.org/time_series/2020-10-06/'
    datafile = url + 'time_series_60min_singleindex.csv'
    df_all = pd.read_csv(datafile, index_col='utc_timestamp', parse_dates=True, low_memory=False)
    df_all.head()

In [18]:
if DOWNLOAD:
    # Download hourly data from OPSD website
    url = 'https://data.open-power-system-data.org/time_series/2020-10-06/'
    datafile = url + 'weather_data.csv'
    df_weather_data = pd.read_csv(datafile, index_col='utc_timestamp', parse_dates=True, low_memory=False)
    df_weather_data.head()

In [19]:
[x for x in os.listdir('.') if x.endswith('csv')]

['weather_data.csv',
 'de_data.csv',
 'time_series_30min_singleindex.csv',
 'de_hourly_power_and_weather.csv',
 'time_series_60min_singleindex.csv',
 'time_series_15min_singleindex.csv']

In [8]:
df_all = pd.read_csv('time_series_60min_singleindex.csv', index_col='utc_timestamp', parse_dates=True, low_memory=False)
df_all.head(1)

,cet_cest_timestamp,AT_load_actual_entsoe_transparency,AT_load_forecast_entsoe_transparency,AT_price_day_ahead,AT_solar_generation_actual,AT_wind_onshore_generation_actual,BE_load_actual_entsoe_transparency,BE_load_forecast_entsoe_transparency,BE_solar_generation_actual,BE_wind_generation_actual,...,SI_load_actual_entsoe_transparency,SI_load_forecast_entsoe_transparency,SI_solar_generation_actual,SI_wind_onshore_generation_actual,SK_load_actual_entsoe_transparency,SK_load_forecast_entsoe_transparency,SK_solar_generation_actual,SK_wind_onshore_generation_actual,UA_load_actual_entsoe_transparency,UA_load_forecast_entsoe_transparency
utc_timestamp,,,,,,,,,,,,,,,,,,,,,
2014-12-31 23:00:00+00:00,2015-01-01T00:00:00+0100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
import pandas as pd
import re

def get_available_countries(power_file='time_series_60min_singleindex.csv', weather_file='weather_data.csv'):
    """
    Возвращает пересечение стран, доступных и в энергетических, и в погодных данных.
    Также выводит отдельно списки по каждому файлу.
    """
    # Загрузка только заголовков (без данных) — быстро и экономно по памяти
    power_cols = pd.read_csv(power_file, nrows=0).columns.tolist()
    weather_cols = pd.read_csv(weather_file, nrows=0).columns.tolist()

    # Извлекаем двухбуквенные коды стран из имён колонок (формат: XX_...)
    def extract_country_codes(col_names):
        codes = set()
        for col in col_names:
            match = re.match(r'^([A-Z]{2})_', col)
            if match:
                codes.add(match.group(1))
        return sorted(codes)

    power_countries = extract_country_codes(power_cols)
    weather_countries = extract_country_codes(weather_cols)
    common_countries = sorted(set(power_countries) & set(weather_countries))

    print("🌍 Страны в энергетических данных:", power_countries)
    print("🌦️  Страны в погодных данных:     ", weather_countries)
    print("✅ Страны, доступные в ОБОИХ:    ", common_countries)

    return common_countries, power_countries, weather_countries

# Пример использования
common, power, weather = get_available_countries()

🌍 Страны в энергетических данных: ['AT', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GB', 'GR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'ME', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'UA']
🌦️  Страны в погодных данных:      ['AT', 'BE', 'BG', 'CH', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GB', 'GR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'NL', 'NO', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK']
✅ Страны, доступные в ОБОИХ:     ['AT', 'BE', 'BG', 'CH', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GB', 'GR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'NL', 'NO', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK']


In [10]:
df_weather = pd.read_csv('weather_data.csv', index_col='utc_timestamp', parse_dates=True, low_memory=False)
df_weather.head(1)

,AT_temperature,AT_radiation_direct_horizontal,AT_radiation_diffuse_horizontal,BE_temperature,BE_radiation_direct_horizontal,BE_radiation_diffuse_horizontal,BG_temperature,BG_radiation_direct_horizontal,BG_radiation_diffuse_horizontal,CH_temperature,...,RO_radiation_diffuse_horizontal,SE_temperature,SE_radiation_direct_horizontal,SE_radiation_diffuse_horizontal,SI_temperature,SI_radiation_direct_horizontal,SI_radiation_diffuse_horizontal,SK_temperature,SK_radiation_direct_horizontal,SK_radiation_diffuse_horizontal
utc_timestamp,,,,,,,,,,,,,,,,,,,,,
1980-01-01 00:00:00+00:00,-3.64,0.0,0.0,-0.72,0.0,0.0,4.664,0.0,0.0,-6.287,...,0.0,-3.945,0.0,0.0,-3.055,0.0,0.0,-4.648,0.0,0.0


In [6]:
def extract_country(df_all, country_code, year_min=None, year_max=None):
    """Extract data for a single country"""
    
    # List of columns to extract
    columns = [col for col in df_all.columns if col.startswith(country_code)]
    
    # Extract columns and remove country codes from column labels
    columns_map = {col : col[3:] for col in columns}
    df_out = df_all[columns].rename(columns=columns_map)
    
    # Exclude years outside of specified range, if any
    if year_min is not None:
        df_out = df_out[df_out.index.year >= year_min]
    if year_max is not None:
        df_out = df_out[df_out.index.year <= year_max]
        
    return df_out
df_hrly = extract_country(df_all, country_code='DE', year_min=2015, year_max=2019)
df_hrly

,load_actual_entsoe_transparency,load_forecast_entsoe_transparency,solar_capacity,solar_generation_actual,solar_profile,wind_capacity,wind_generation_actual,wind_profile,wind_offshore_capacity,wind_offshore_generation_actual,...,tennet_load_actual_entsoe_transparency,tennet_load_forecast_entsoe_transparency,tennet_solar_generation_actual,tennet_wind_generation_actual,tennet_wind_offshore_generation_actual,tennet_wind_onshore_generation_actual,transnetbw_load_actual_entsoe_transparency,transnetbw_load_forecast_entsoe_transparency,transnetbw_solar_generation_actual,transnetbw_wind_onshore_generation_actual
utc_timestamp,,,,,,,,,,,,,,,,,,,,,
2015-01-01 00:00:00+00:00,41151.0,39723.0,37248.0,NaN,NaN,27913.0,8852.0,0.3171,667.0,517.0,...,13841.0,13362.0,NaN,3866.0,469.0,3398.0,5307.0,4703.0,NaN,5.0
2015-01-01 01:00:00+00:00,40135.0,38813.0,37248.0,NaN,NaN,27913.0,9054.0,0.3244,667.0,514.0,...,13267.0,12858.0,NaN,3974.0,466.0,3508.0,5087.0,4562.0,NaN,7.0
2015-01-01 02:00:00+00:00,39106.0,38490.0,37248.0,NaN,NaN,27913.0,9070.0,0.3249,667.0,518.0,...,12702.0,12611.0,NaN,4194.0,470.0,3724.0,4906.0,4517.0,NaN,8.0
2015-01-01 03:00:00+00:00,38765.0,38644.0,37248.0,NaN,NaN,27913.0,9163.0,0.3283,667.0,520.0,...,12452.0,12490.0,NaN,4446.0,473.0,3973.0,4865.0,4601.0,NaN,11.0
2015-01-01 04:00:00+00:00,38941.0,38773.0,37248.0,NaN,NaN,27913.0,9231.0,0.3307,667.0,520.0,...,12454.0,12464.0,NaN,4671.0,474.0,4198.0,4685.0,4519.0,NaN,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-31 19:00:00+00:00,47493.0,54131.0,NaN,0.0,NaN,NaN,8875.0,NaN,NaN,873.0,...,14753.0,15838.0,0.0,1764.0,274.0,1490.0,5676.0,5584.0,0.0,126.0
2019-12-31 20:00:00+00:00,45842.0,51764.0,NaN,0.0,NaN,NaN,7652.0,NaN,NaN,704.0,...,14018.0,15145.0,0.0,1384.0,151.0,1233.0,5507.0,5564.0,0.0,109.0
2019-12-31 21:00:00+00:00,45501.0,50569.0,NaN,0.0,NaN,NaN,7283.0,NaN,NaN,575.0,...,14064.0,15275.0,0.0,1223.0,57.0,1166.0,5530.0,5522.0,0.0,152.0


In [7]:
def transform_dataframe(df, cols_map):
    # Rename columns for convenience
    df = df[list(cols_map.keys())].rename(columns=cols_map)
    # Convert from MW to GW
    df = df / 1000
    df = df.resample('D').sum(min_count=24)
    df = df.rename_axis('Date')
    df.index = df.index.strftime('%Y-%m-%d')
    return df

cols_map = {'load_actual_entsoe_transparency' : 'Consumption',
            'wind_generation_actual' : 'Wind',
            'solar_generation_actual' : 'Solar'}
df_daily = transform_dataframe(df_hrly, cols_map)

# Compute wind + solar generation
df_daily['Wind+Solar'] = df_daily[['Wind', 'Solar']].sum(axis=1, skipna=False)
df_daily.to_csv('de_data.csv')
df_daily.head()

,Consumption,Wind,Solar,Wind+Solar
Date,,,,
2015-01-01,1088.317,325.165,NaN,NaN
2015-01-02,1246.588,603.554,7.757,611.311
2015-01-03,1117.554,462.955,7.237,470.192
2015-01-04,1081.980,385.023,19.982,405.005
2015-01-05,1325.920,216.540,26.522,243.062


In [11]:
import pandas as pd

# Загрузка данных
df_all = pd.read_csv('time_series_60min_singleindex.csv', index_col='utc_timestamp', parse_dates=True, low_memory=False)
df_weather = pd.read_csv('weather_data.csv', index_col='utc_timestamp', parse_dates=True, low_memory=False)

def extract_country(df, country_code, year_min=None, year_max=None):
    """Извлечь данные только для указанной страны (по префиксу колонок)"""
    cols = [col for col in df.columns if col.startswith(country_code)]
    if not cols:
        raise ValueError(f"Нет колонок с префиксом '{country_code}' в переданном DataFrame.")
    
    # Убираем префикс страны из названий колонок
    cols_map = {col: col[len(country_code)+1:] for col in cols}  # Например: 'DE_wind' → 'wind'
    df_out = df[cols].rename(columns=cols_map)
    
    # Фильтрация по годам, если заданы границы
    if year_min is not None:
        df_out = df_out[df_out.index.year >= year_min]
    if year_max is not None:
        df_out = df_out[df_out.index.year <= year_max]
        
    return df_out

# Извлекаем энергетические данные по Германии (часовой шаг, 2015–2019)
df_power_hrly = extract_country(df_all, country_code='DE', year_min=2015, year_max=2019)

# Извлекаем погодные данные по Германии (часовой шаг, за тот же период)
df_weather_hrly = extract_country(df_weather, country_code='DE', year_min=2015, year_max=2019)

# Приведение к совместному временному диапазону (пересечение)
common_index = df_power_hrly.index.intersection(df_weather_hrly.index)
df_power_hrly = df_power_hrly.loc[common_index]
df_weather_hrly = df_weather_hrly.loc[common_index]

# Преобразуем энергетические колонки (перевод из МВт в ГВт — если нужно)
power_cols_map = {
    'load_actual_entsoe_transparency': 'Consumption',
    'wind_generation_actual': 'Wind',
    'solar_generation_actual': 'Solar'
}

# Оставляем только нужные колонки и переименовываем
df_power = df_power_hrly[list(power_cols_map.keys())].rename(columns=power_cols_map) / 1000  # МВт → ГВт

# Добавляем суммарную генерацию Wind + Solar (с сохранением NaN, если одна из колонок NaN)
df_power['Wind+Solar'] = df_power[['Wind', 'Solar']].sum(axis=1, skipna=False)

# Объединяем с погодными данными
df_combined = pd.concat([df_power, df_weather_hrly], axis=1)

# Сохраняем в CSV
df_combined.to_csv('de_hourly_power_and_weather.csv', index_label='utc_timestamp')

# Показываем первые строки
df_combined.head()

                           Consumption   Wind  Solar  Wind+Solar  temperature  \
utc_timestamp                                                                   
2015-01-01 00:00:00+00:00       41.151  8.852    NaN         NaN       -0.981   
2015-01-01 01:00:00+00:00       40.135  9.054    NaN         NaN       -1.035   
2015-01-01 02:00:00+00:00       39.106  9.070    NaN         NaN       -1.109   
2015-01-01 03:00:00+00:00       38.765  9.163    NaN         NaN       -1.166   
2015-01-01 04:00:00+00:00       38.941  9.231    NaN         NaN       -1.226   

                           radiation_direct_horizontal  \
utc_timestamp                                            
2015-01-01 00:00:00+00:00                          0.0   
2015-01-01 01:00:00+00:00                          0.0   
2015-01-01 02:00:00+00:00                          0.0   
2015-01-01 03:00:00+00:00                          0.0   
2015-01-01 04:00:00+00:00                          0.0   

                         

In [12]:
df_combined.head()

,Consumption,Wind,Solar,Wind+Solar,temperature,radiation_direct_horizontal,radiation_diffuse_horizontal
utc_timestamp,,,,,,,
2015-01-01 00:00:00+00:00,41.151,8.852,NaN,NaN,-0.981,0.0,0.0
2015-01-01 01:00:00+00:00,40.135,9.054,NaN,NaN,-1.035,0.0,0.0
2015-01-01 02:00:00+00:00,39.106,9.070,NaN,NaN,-1.109,0.0,0.0
2015-01-01 03:00:00+00:00,38.765,9.163,NaN,NaN,-1.166,0.0,0.0
2015-01-01 04:00:00+00:00,38.941,9.231,NaN,NaN,-1.226,0.0,0.0


In [13]:
import pandas as pd

def prepare_country_data(
    country_code: str,
    power_file: str = 'time_series_60min_singleindex.csv',
    weather_file: str = 'weather_data.csv',
    year_min: int = 2015,
    year_max: int = 2019
):
    """
    Подготавливает почасовые данные по энергетике и погоде для заданной страны.
    
    Параметры:
        country_code (str): Код страны (например, 'DE', 'FR', 'PL').
        power_file (str): Путь к файлу с энергетическими данными.
        weather_file (str): Путь к файлу с погодными данными.
        year_min, year_max (int): Годы для фильтрации.
    
    Возвращает:
        pd.DataFrame: Объединённый DataFrame с часовой частотой.
    """
    
    # Загрузка данных
    df_power = pd.read_csv(power_file, index_col='utc_timestamp', parse_dates=True, low_memory=False)
    df_weather = pd.read_csv(weather_file, index_col='utc_timestamp', parse_dates=True, low_memory=False)

    def extract_by_country(df, prefix):
        """Извлекает колонки по префиксу страны и убирает его из имён."""
        cols = [col for col in df.columns if col.startswith(prefix)]
        if not cols:
            raise ValueError(f"Нет колонок с префиксом '{prefix}' в DataFrame.")
        clean_names = {col: col[len(prefix)+1:] for col in cols}
        return df[cols].rename(columns=clean_names)

    # Извлечение данных для страны
    power = extract_by_country(df_power, country_code)
    weather = extract_by_country(df_weather, country_code)

    # Фильтрация по годам
    mask = (power.index.year >= year_min) & (power.index.year <= year_max)
    power = power[mask]
    weather = weather[weather.index.isin(power.index)]  # синхронизация по времени
    power = power[power.index.isin(weather.index)]      # гарантируем совпадение индексов

    # Отбор и переименование энергетических колонок
    power_map = {
        'load_actual_entsoe_transparency': 'Consumption',   # потребление, ГВт
        'wind_generation_actual': 'Wind',                   # ветровая генерация
        'solar_generation_actual': 'Solar',                 # солнечная генерация
        # Можете добавить другие, если есть:
        # 'total_load_actual': 'Total_Load',
        # 'wind_onshore_generation_actual': 'Wind_Onshore',
        # 'wind_offshore_generation_actual': 'Wind_Offshore',
        # 'solar_rooftop_generation_actual': 'Solar_Rooftop',
    }
    
    # Оставить только доступные колонки
    available_power_cols = {k: v for k, v in power_map.items() if k in power.columns}
    df_power_clean = power[list(available_power_cols.keys())].rename(columns=available_power_cols) / 1000  # МВт → ГВт

    # Добавляем суммарную ВИЭ-генерацию
    if 'Wind' in df_power_clean.columns and 'Solar' in df_power_clean.columns:
        df_power_clean['Wind+Solar'] = df_power_clean[['Wind', 'Solar']].sum(axis=1, skipna=False)

    # Объединение с погодой
    df_final = pd.concat([df_power_clean, weather], axis=1)

    # Сохранение
    output_file = f'{country_code.lower()}_hourly_power_and_weather.csv'
    df_final.to_csv(output_file, index_label='utc_timestamp')
    print(f"✅ Данные сохранены в: {output_file}")
    
    return df_final

# Пример использования
df_de = prepare_country_data(country_code='DE', year_min=2015, year_max=2019)
df_de.head()

✅ Данные сохранены в: de_hourly_power_and_weather.csv


,Consumption,Wind,Solar,Wind+Solar,temperature,radiation_direct_horizontal,radiation_diffuse_horizontal
utc_timestamp,,,,,,,
2015-01-01 00:00:00+00:00,41.151,8.852,NaN,NaN,-0.981,0.0,0.0
2015-01-01 01:00:00+00:00,40.135,9.054,NaN,NaN,-1.035,0.0,0.0
2015-01-01 02:00:00+00:00,39.106,9.070,NaN,NaN,-1.109,0.0,0.0
2015-01-01 03:00:00+00:00,38.765,9.163,NaN,NaN,-1.166,0.0,0.0
2015-01-01 04:00:00+00:00,38.941,9.231,NaN,NaN,-1.226,0.0,0.0


In [21]:
# Пример для Германии (DE)
prefix = 'DE'
candidate_speed = None
candidate_dir = None

for col in df_weather.columns:
    if col.startswith(prefix):
        clean_name = col[len(prefix)+1:]  # убираем "DE_"
        if 'speed' in clean_name or 'windspeed' in clean_name:
            candidate_speed = col
        if 'direction' in clean_name or 'dir' in clean_name:
            candidate_dir = col

print("Предполагаемая колонка скорости ветра:", candidate_speed)
print("Предполагаемая колонка направления:", candidate_dir)

# Проверим статистику
if candidate_speed:
    print("\nСтатистика по скорости ветра:")
    print(df_weather[candidate_speed].describe())
if candidate_dir:
    print("\nСтатистика по направлению ветра:")
    print(df_weather[candidate_dir].describe())

Предполагаемая колонка скорости ветра: None
Предполагаемая колонка направления: DE_radiation_direct_horizontal

Статистика по направлению ветра:
count    350640.000000
mean         77.780040
std         153.118508
min           0.000000
25%           0.000000
50%           0.369850
75%          70.400900
max         849.984900
Name: DE_radiation_direct_horizontal, dtype: float64
